In [3]:

import time
import gc
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# 1. Загрузка корпуса
df = pd.read_csv("data/lenta_balanced.csv").dropna(subset=["topic", "text"])
df = df.sample(n=min(10000, len(df)), random_state=42)

df["text"] = (
    df["title"].fillna("").astype(str) + ". " +
    df["text"].astype(str)
)

encoder = LabelEncoder()
df["labels"] = encoder.fit_transform(df["topic"])

train, test = train_test_split(
    df[["text", "labels"]],
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

train_ds = Dataset.from_pandas(train.reset_index(drop=True))
test_ds = Dataset.from_pandas(test.reset_index(drop=True))

models = {
    "BERT": "bert-base-multilingual-cased",
    "DistilBERT": "distilbert-base-multilingual-cased"
}

results = []

# 2. Сравнительное обучение
for name, model_name in models.items():
    print(f"\n--- {name} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=128
        )

    tr = train_ds.map(tokenize, batched=True, remove_columns=["text"])
    te = test_ds.map(tokenize, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(encoder.classes_)
    )

    args = TrainingArguments(
        output_dir=f"./results_{name}",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr
    )

    start = time.perf_counter()
    trainer.train()
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    prediction = trainer.predict(te)
    inference_time = time.perf_counter() - start

    y_true = prediction.label_ids
    y_pred = np.argmax(prediction.predictions, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Macro F1": round(macro_f1, 4),
        "Training (sec)": round(train_time, 2),
        "Inference (sec)": round(inference_time, 2),
        "Samples/sec": round(len(test) / inference_time, 2)
    })

    print(results[-1])

    del trainer, model, tokenizer, tr, te
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 3. Итоговое сравнение
print("\n========== COMPARISON ==========")
print(pd.DataFrame(results).to_string(index=False))


d:\magistr\diplom\.dipl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nikit\AppData\Local\Temp\ipykernel_2060\950726381.py:19: DtypeWarning: Columns (3: topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/lenta_balanced.csv").dropna(subset=["topic", "text"])



--- BERT ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 309.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Step,Training Loss
500,1.289551


{'Model': 'BERT', 'Accuracy': 0.752, 'Macro F1': 0.4369, 'Training (sec)': 61.05, 'Inference (sec)': 2.57, 'Samples/sec': 777.64}

--- DistilBERT ---


d:\magistr\diplom\.dipl\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikit\.cache\huggingface\hub\models--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 2000/2000 [00:00<00:00, 6251.23 examples/s]
d:\magistr\diplom\.dipl\Lib\site-packages\hug

Step,Training Loss
500,1.472536


{'Model': 'DistilBERT', 'Accuracy': 0.7085, 'Macro F1': 0.3764, 'Training (sec)': 30.45, 'Inference (sec)': 1.27, 'Samples/sec': 1580.79}

========== COMPARISON ==========
     Model  Accuracy  Macro F1  Training (sec)  Inference (sec)  Samples/sec
      BERT    0.7520    0.4369           61.05             2.57       777.64
DistilBERT    0.7085    0.3764           30.45             1.27      1580.79


In [2]:
import  torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4060


In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA-сборка PyTorch:", torch.version.cuda)
print("CUDA доступна:", torch.cuda.is_available())

try:
    torch.cuda.init()
except Exception as e:
    print("Ошибка инициализации CUDA:", repr(e))

Python: d:\magistr\diplom\.dipl\Scripts\python.exe
PyTorch: 2.15.0+cu134
CUDA-сборка PyTorch: 13.4
CUDA доступна: True
